# 🧠 UnblurNews — ModernBERT Multi-Task Training

Fine-tunes **ModernBERT-base** on three simultaneous classification tasks:

| Head | Task | Classes |
|------|------|---------|
| `clickbait_head` | Clickbait detection | 0 = Not clickbait · 1 = Clickbait |
| `leaning_head` | Political leaning | 0 = Left · 1 = Center · 2 = Right |
| `sentiment_head` | Sentiment | 0 = Negative · 1 = Neutral · 2 = Positive |

**Runtime:** GPU → T4 (Runtime → Change runtime type → GPU)  
**Expected total time:** ~2–4 hours on T4  
**Output:** `unblur_model.zip` — ready to drop into `unblur/backend/models/`

In [ ]:
# @title ⚙️ Install Dependencies
%%capture
!pip install transformers datasets accelerate scikit-learn tqdm pandas huggingface_hub

In [ ]:
# @title 📦 Imports & Configuration
import os, time, math, random, json, zipfile, shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device ───────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    torch.cuda.manual_seed(SEED)
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — training on CPU will be very slow.")
    print("    Go to: Runtime → Change runtime type → GPU (T4)")

# ── Hyperparameters ───────────────────────────────────────────────
MODEL_NAME      = "answerdotai/ModernBERT-base"
MAX_LEN_SHORT   = 128    # clickbait & sentiment (headlines / tweets)
MAX_LEN_LONG    = 256    # political leaning (article paragraphs)
BATCH_SIZE      = 32     # reduce to 16 if you get CUDA out-of-memory
LR              = 2e-5
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.06
MAX_GRAD_NORM   = 1.0
EPOCHS_TASK     = 3      # epochs per individual-head training phase
EPOCHS_MULTI    = 2      # epochs for the final multi-task phase
MAX_SAMPLES     = 30_000 # cap per task to keep runtime reasonable
USE_FP16        = device.type == "cuda"

# ── Paths ─────────────────────────────────────────────────────────
OUTPUT_DIR      = "/content/unblur_model"
CKPT_DIR        = "/content/checkpoints"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)

print(f"\nConfig: model={MODEL_NAME} | batch={BATCH_SIZE} | fp16={USE_FP16}")
print(f"Output: {OUTPUT_DIR}")

## 📊 Section 1 — Load Datasets

Each loader tries multiple sources and falls back gracefully.

In [ ]:
# @title 📰 1A — Clickbait Dataset  (christophsonntag/clickbait)
from datasets import load_dataset

def load_clickbait(max_samples=MAX_SAMPLES):
    print("Loading clickbait data...")
    ds = load_dataset("christophsonntag/clickbait", trust_remote_code=True)
    split = list(ds.keys())[0]
    data  = ds[split]
    cols  = data.column_names
    print(f"  Columns: {cols}  |  Rows: {len(data)}")

    if "clickbait" in cols and "not_clickbait" in cols:
        cb  = [str(t) for t in data["clickbait"]     if t and str(t).strip()]
        ncb = [str(t) for t in data["not_clickbait"] if t and str(t).strip()]
        texts  = cb + ncb
        labels = [1] * len(cb) + [0] * len(ncb)
    else:
        tc = next((c for c in ["text","headline","title","article"] if c in cols), cols[0])
        lc = next((c for c in ["label","clickbait","is_clickbait","class"] if c in cols), None)
        if lc is None:
            raise ValueError(f"Cannot find label column in {cols}")
        texts  = [str(t) for t in data[tc]]
        labels = [int(l) for l in data[lc]]

    # Balance & cap
    half = (max_samples or len(texts)) // 2
    pos  = [(t, 1) for t, l in zip(texts, labels) if l == 1][:half]
    neg  = [(t, 0) for t, l in zip(texts, labels) if l == 0][:half]
    data_pairs = pos + neg
    random.shuffle(data_pairs)
    texts, labels = zip(*data_pairs)
    texts, labels = list(texts), list(labels)

    print(f"  ✓ {len(texts)} samples  |  clickbait={labels.count(1)}  normal={labels.count(0)}")
    print(f"  Example: '{texts[0][:80]}'")
    return texts, labels

cb_texts, cb_labels = load_clickbait()

In [ ]:
# @title 🗳️ 1B — Political Leaning Dataset  (multiple sources with fallback)
# Labels: 0=Left  1=Center  2=Right

# ── Label string → integer mapping ──────────────────────────────
LEAN_MAP = {
    "left":0,"left-center":0,"lean left":0,"lean-left":0,"liberal":0,
    "progressive":0,"left wing":0,"leftwing":0,"left_center":0,
    "center":1,"centre":1,"neutral":1,"moderate":1,"centrist":1,
    "center-left":1,"center-right":1,"center left":1,"center right":1,"balanced":1,
    "right":2,"right-center":2,"lean right":2,"lean-right":2,"conservative":2,
    "republican":2,"right wing":2,"rightwing":2,"right_center":2,
}

def _map_lean_labels(raw):
    out = []
    for r in raw:
        k = str(r).lower().strip()
        if k in LEAN_MAP:            out.append(LEAN_MAP[k])
        elif k.isdigit() and int(k) in (0,1,2): out.append(int(k))
        else: return None   # unmappable
    return out

TEXT_COLS  = ["text","article","content","headline","title","sentence","body","statement"]
LABEL_COLS = ["label","bias","leaning","category","class","political_leaning",
              "media_bias","orientation","political_bias","bias_label"]

def _try_hf(name, extra_configs=None):
    """Try loading a HuggingFace dataset with flexible column detection."""
    configs = extra_configs or [None]
    for cfg in configs:
        try:
            ds = load_dataset(name, cfg, trust_remote_code=True) if cfg else load_dataset(name, trust_remote_code=True)
            # Merge all splits
            texts, labels = [], []
            for sname, split in ds.items():
                cols = split.column_names
                tc = next((c for c in TEXT_COLS  if c in cols), None)
                lc = next((c for c in LABEL_COLS if c in cols), None)
                if not tc or not lc:
                    continue
                raw = [str(r) for r in split[lc]]
                mapped = _map_lean_labels(raw)
                if mapped is None:
                    unique = set(raw[:20])
                    print(f"    [{name}] Unknown labels: {unique}")
                    continue
                texts  += [str(t) for t in split[tc]]
                labels += mapped
            if len(texts) >= 200:
                return texts, labels
        except Exception as e:
            pass
    return None, None

# ── Hardcoded fallback (25 examples/class) ───────────────────────
FALLBACK_LEFT = [
    "Republican tax cuts for the wealthy are devastating working families and widening inequality.",
    "Universal healthcare is a human right and the government must act to guarantee it for all.",
    "The GOP's refusal to address climate change is negligence against future generations.",
    "Corporate greed and union-busting are the root causes of wage stagnation for workers.",
    "Systemic racism embedded in our institutions demands bold progressive policy reform.",
    "Billionaires should pay their fair share — a wealth tax is long overdue in America.",
    "Workers' rights are under attack by Republican governors pushing anti-union legislation.",
    "Medicare for All would save millions of lives and trillions of dollars over the long term.",
    "The fossil fuel industry has spent decades lying about climate change to protect profits.",
    "Student debt cancellation would lift millions out of poverty and stimulate the economy.",
    "Reproductive rights are human rights — abortion must remain safe, legal, and accessible.",
    "The minimum wage must be raised to $15 or higher to reflect the true cost of living.",
    "Dark money in politics is corrupting our democracy and must be abolished immediately.",
    "Voter suppression laws are designed to disenfranchise Black and brown Americans.",
    "The war on drugs has been a racist policy that devastated communities of color.",
    "Child poverty can be ended with expanded social safety net programs — the will is lacking.",
    "Public education is underfunded while charter schools siphon money from communities.",
    "Gun violence is a public health crisis requiring bold legislative action from Congress.",
    "Trickle-down economics has never worked and has only enriched the billionaire class.",
    "We must abolish the Electoral College and move to a true national popular vote.",
    "Immigrants built this country and xenophobic policies hurt our economy and values.",
    "The prison-industrial complex must be dismantled and replaced with rehabilitation.",
    "Corporate media is a mouthpiece for the wealthy elite, not the working people.",
    "Big Pharma's profit motive kills people — we need price controls on prescription drugs.",
    "The climate crisis demands a Green New Deal and massive public investment in clean energy.",
]
FALLBACK_CENTER = [
    "The Federal Reserve held interest rates steady, citing stable inflation and moderate growth.",
    "Both parties failed to reach a compromise on the budget, leaving the issue unresolved.",
    "Economists are divided on the long-term effects of the proposed trade policy changes.",
    "The bipartisan infrastructure bill passed with support from members of both parties.",
    "Analysts say the economy shows mixed signals, with job growth offset by rising prices.",
    "Negotiators from both parties are working toward a compromise on immigration legislation.",
    "Independent voters remain skeptical of proposals from both the left and the right.",
    "The nonpartisan budget office projected modest growth under the current fiscal policy.",
    "Representatives from across the aisle expressed concern about rising national debt.",
    "The court's decision drew criticism from both progressive and conservative groups.",
    "Experts say addressing climate change requires realistic policy, not ideological extremism.",
    "Polling shows Americans are frustrated with gridlock in Washington from both parties.",
    "The technology company faced scrutiny from regulators appointed by both parties.",
    "Healthcare reform requires addressing costs while maintaining access — a difficult balance.",
    "Both candidates have proposed competing economic plans with significant trade-offs.",
    "Civil liberties groups expressed concern about surveillance legislation backed by both parties.",
    "The trade agreement has supporters and critics across the political spectrum.",
    "Infrastructure investment enjoys broad public support regardless of political affiliation.",
    "Fiscal responsibility and social investment are both important goals for policymakers.",
    "Campaign finance reform has advocates in both the Democratic and Republican parties.",
    "The unemployment rate fell slightly, though analysts caution against over-interpreting.",
    "The report called for evidence-based immigration policy rather than partisan rhetoric.",
    "The study found that policies from both administrations had pros and cons for growth.",
    "Experts say the most effective criminal justice policies combine accountability and rehab.",
    "The commission released its report calling for reforms to address concerns from both sides.",
]
FALLBACK_RIGHT = [
    "Biden's open-border policies have created an unprecedented national security crisis.",
    "The radical left's socialist agenda is destroying the American economy and way of life.",
    "Tax cuts for job creators stimulate growth, create jobs, and ultimately benefit all Americans.",
    "Second Amendment rights must be protected against the government's unconstitutional overreach.",
    "The mainstream media is covering up the truth — the corrupt Democrat machine must be stopped.",
    "Critical race theory is indoctrinating our children with anti-American propaganda in schools.",
    "Law enforcement deserves our full support — defunding the police endangers every community.",
    "America-first trade policies protect our workers from China's economic and military warfare.",
    "The radical green agenda will destroy our energy independence and kill millions of jobs.",
    "Parents have the right to know what is being taught to their children — no more woke ideology.",
    "Election integrity is non-negotiable — we must ensure every legal vote is counted.",
    "Free market solutions — not government mandates — are the answer to rising healthcare costs.",
    "The Constitution is the supreme law of the land and must be defended against leftist attacks.",
    "America's energy independence was achieved under Republican leadership and must be preserved.",
    "Woke corporations are pushing an extreme political agenda on unwilling American consumers.",
    "The weaponization of the DOJ against conservatives is an unprecedented abuse of power.",
    "Religious liberty is under attack from the radical secularist agenda of the Democrat Party.",
    "Strong borders and controlled immigration protect American workers and communities.",
    "Inflation is the direct result of reckless Democrat spending that must be reversed now.",
    "School choice gives every child — regardless of zip code — access to a quality education.",
    "The military has been weakened by woke ideology when it should be focused on combat readiness.",
    "Judicial activism by left-wing judges is bypassing the will of the American people.",
    "Small businesses are being crushed by excessive government regulation and taxation.",
    "Traditional family values are the foundation of a strong, prosperous, and free society.",
    "Government dependency is destroying communities — we need to restore personal responsibility.",
]

def load_political(max_samples=MAX_SAMPLES):
    print("Loading political leaning data...")
    texts, labels = None, None

    # Ordered list of datasets to try
    attempts = [
        ("cajcodes/political-news-dataset", None),
        ("valurank/political-news-article-bias", None),
        ("newsmediabias/BEAD", None),
        ("ARTeLab/loco", None),
    ]
    for name, cfg in attempts:
        print(f"  Trying {name}...")
        t, l = _try_hf(name, [cfg] if cfg else None)
        if t and len(t) >= 200:
            print(f"  ✓ Loaded {len(t)} samples from {name}")
            texts, labels = t, l
            break
        else:
            print(f"  ✗ Not usable")

    if texts is None:
        print("\n  ⚠️  All remote datasets failed.")
        print("  ⚠️  Using built-in fallback (25 examples/class).")
        print("  ⚠️  For production accuracy, supply a real political dataset!")
        texts  = FALLBACK_LEFT + FALLBACK_CENTER + FALLBACK_RIGHT
        labels = [0]*25 + [1]*25 + [2]*25

    # Balance & cap
    per_class = (max_samples or len(texts)) // 3
    result = []
    for cls in (0, 1, 2):
        pairs = [(t, l) for t, l in zip(texts, labels) if l == cls][:per_class]
        result.extend(pairs)
    random.shuffle(result)
    texts, labels = zip(*result)
    texts, labels = list(texts), list(labels)

    c = {0: labels.count(0), 1: labels.count(1), 2: labels.count(2)}
    print(f"  Final: {len(texts)} samples  |  Left={c[0]}  Center={c[1]}  Right={c[2]}")
    return texts, labels

pol_texts, pol_labels = load_political()

In [ ]:
# @title 😐 1C — Sentiment Dataset  (cardiffnlp/tweet_eval)
# Labels: 0=Negative  1=Neutral  2=Positive

def load_sentiment(max_samples=MAX_SAMPLES):
    print("Loading sentiment data...")
    try:
        ds = load_dataset("cardiffnlp/tweet_eval", "sentiment", trust_remote_code=True)
        texts, labels = [], []
        for sname in ("train", "validation", "test"):
            if sname in ds:
                texts  += [str(t) for t in ds[sname]["text"]]
                labels += [int(l) for l in ds[sname]["label"]]
        print(f"  ✓ tweet_eval/sentiment: {len(texts)} samples")
    except Exception as e:
        print(f"  ✗ tweet_eval failed: {e}")
        print("  Trying amazon_polarity as fallback...")
        ds = load_dataset("amazon_polarity", split="train[:40000]", trust_remote_code=True)
        texts  = [str(t) for t in ds["content"]]
        # 0=negative→0, 1=positive→2 (no neutral class)
        labels = [0 if l == 0 else 2 for l in ds["label"]]
        print(f"  ✓ amazon_polarity fallback: {len(texts)} samples (binary, no neutral)")

    # Balance & cap
    per_class = (max_samples or len(texts)) // 3
    result = []
    for cls in (0, 1, 2):
        pairs = [(t, l) for t, l in zip(texts, labels) if l == cls][:per_class]
        result.extend(pairs)
    random.shuffle(result)
    texts, labels = zip(*result)
    texts, labels = list(texts), list(labels)

    c = {0: labels.count(0), 1: labels.count(1), 2: labels.count(2)}
    print(f"  Final: {len(texts)} samples  |  Neg={c[0]}  Neu={c[1]}  Pos={c[2]}")
    return texts, labels

sent_texts, sent_labels = load_sentiment()

print("\n" + "="*50)
print("Dataset summary")
print("="*50)
print(f"  Clickbait : {len(cb_texts):>6,} samples")
print(f"  Political : {len(pol_texts):>6,} samples")
print(f"  Sentiment : {len(sent_texts):>6,} samples")

## 🔧 Section 2 — Preprocessing & DataLoaders

In [ ]:
# @title Load tokenizer and build DataLoaders

class TextDataset(Dataset):
    """Pre-tokenizes all texts at init for faster training."""
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            max_length=max_length,
            padding="max_length",
            truncation=True,
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      torch.tensor(self.encodings["input_ids"][idx],      dtype=torch.long),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long),
            "labels":         torch.tensor(self.labels[idx],                      dtype=torch.long),
        }

def make_loaders(texts, labels, tokenizer, max_length, train_ratio=0.85):
    """Build balanced train/val DataLoaders."""
    ds = TextDataset(texts, labels, tokenizer, max_length)
    n_train = int(len(ds) * train_ratio)
    n_val   = len(ds) - n_train
    train_ds, val_ds = random_split(ds, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(SEED))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,   shuffle=True,
                              num_workers=2, pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                              num_workers=2, pin_memory=(device.type=="cuda"))
    return train_loader, val_loader

# ── Load tokenizer ────────────────────────────────────────────────
print(f"Loading tokenizer: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✓ Tokenizer ready")

# ── Build DataLoaders ─────────────────────────────────────────────
print("\nTokenizing datasets (may take a few minutes)...")
print("  Clickbait...", end="", flush=True)
cb_train, cb_val     = make_loaders(cb_texts,   cb_labels,   tokenizer, MAX_LEN_SHORT)
print(f" {len(cb_train)} train batches")

print("  Political...", end="", flush=True)
pol_train, pol_val   = make_loaders(pol_texts,  pol_labels,  tokenizer, MAX_LEN_LONG)
print(f" {len(pol_train)} train batches")

print("  Sentiment...", end="", flush=True)
sent_train, sent_val = make_loaders(sent_texts, sent_labels, tokenizer, MAX_LEN_SHORT)
print(f" {len(sent_train)} train batches")

print("\n✓ All DataLoaders ready")

## 🧠 Section 3 — Model Architecture

Single ModernBERT backbone shared by three classification heads.

In [ ]:
# @title MultiHeadModernBERT model definition

class MultiHeadModernBERT(nn.Module):
    """
    ModernBERT backbone + three classification heads.

    Head architecture (shared pattern):
        Dropout → Linear(768→256) → ReLU → Dropout → Linear(256→n_classes)

    Head names match unblur/backend/analyzer.py:
        clickbait_head  (2 classes)
        leaning_head    (3 classes: left / center / right)
        sentiment_head  (3 classes: neg / neu / pos)
    """

    def __init__(self, model_name=MODEL_NAME, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size  # 768 for ModernBERT-base

        def _head(n_classes):
            return nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(hidden, 256),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(256, n_classes),
            )

        self.clickbait_head = _head(2)
        self.leaning_head   = _head(3)   # political leaning
        self.sentiment_head = _head(3)

    def _cls(self, input_ids, attention_mask):
        """Run backbone and return [CLS] token representation."""
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]  # shape: (batch, hidden_size)

    def forward(self, input_ids, attention_mask, task="all"):
        cls = self._cls(input_ids, attention_mask)
        if task == "clickbait":
            return self.clickbait_head(cls)
        elif task == "leaning":
            return self.leaning_head(cls)
        elif task == "sentiment":
            return self.sentiment_head(cls)
        elif task == "all":
            return {
                "clickbait": self.clickbait_head(cls),
                "leaning":   self.leaning_head(cls),
                "sentiment": self.sentiment_head(cls),
            }
        raise ValueError(f"Unknown task: {task!r}")

    def save_ckpt(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "model_state_dict": self.state_dict(),
            "config": {"model_name": MODEL_NAME},
        }, path)

    @classmethod
    def load_ckpt(cls, path):
        ckpt  = torch.load(path, map_location=device, weights_only=False)
        model = cls(model_name=ckpt["config"].get("model_name", MODEL_NAME))
        model.load_state_dict(ckpt["model_state_dict"])
        return model.to(device)

print(f"Downloading backbone: {MODEL_NAME} ...")
model = MultiHeadModernBERT().to(device)

total = sum(p.numel() for p in model.parameters())
train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model ready  |  total params: {total/1e6:.1f}M  |  trainable: {train/1e6:.1f}M")

## 🏋️ Section 4 — Training

**Strategy:** Train each head sequentially (backbone updates each time), then joint multi-task fine-tuning.

Mixed-precision (fp16) is used automatically when a GPU is available.

In [ ]:
# @title Training utilities (train_epoch, evaluate, train_task)

criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, scheduler, task, scaler):
    model.train()
    total_loss, all_preds, all_true = 0.0, [], []

    pbar = tqdm(loader, desc=f"  [{task}] train", leave=False, ncols=90)
    for batch in pbar:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        if scaler:
            with autocast():
                logits = model(ids, mask, task=task)
                loss   = criterion(logits, lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(ids, mask, task=task)
            loss   = criterion(logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

        scheduler.step()
        total_loss += loss.item()
        all_preds  += logits.argmax(dim=-1).cpu().tolist()
        all_true   += lbls.cpu().tolist()
        pbar.set_postfix(loss=f"{loss.item():.3f}")

    return total_loss / len(loader), accuracy_score(all_true, all_preds)


@torch.no_grad()
def evaluate(model, loader, task):
    model.eval()
    total_loss, all_preds, all_true = 0.0, [], []

    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)

        if USE_FP16:
            with autocast():
                logits = model(ids, mask, task=task)
        else:
            logits = model(ids, mask, task=task)

        total_loss += criterion(logits, lbls).item()
        all_preds  += logits.argmax(dim=-1).cpu().tolist()
        all_true   += lbls.cpu().tolist()

    return total_loss / len(loader), accuracy_score(all_true, all_preds), all_preds, all_true


def train_task(model, task, train_loader, val_loader, n_epochs=EPOCHS_TASK):
    """Full training loop for one task. Returns model with best val accuracy."""
    print(f"\n{'='*55}")
    print(f"  Training: {task.upper()} HEAD  ({n_epochs} epochs)")
    print(f"{'='*55}")

    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps  = len(train_loader) * n_epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = GradScaler() if USE_FP16 else None

    best_acc  = 0.0
    best_path = f"{CKPT_DIR}/best_{task}.pt"

    for epoch in range(1, n_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc   = train_epoch(model, train_loader, optimizer, scheduler, task, scaler)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, task)
        mins = (time.time() - t0) / 60

        marker = " ★" if vl_acc > best_acc else ""
        print(f"  Epoch {epoch}/{n_epochs}  "
              f"train {tr_loss:.4f}/{tr_acc:.2%}  "
              f"val {vl_loss:.4f}/{vl_acc:.2%}  "
              f"{mins:.1f}m{marker}")

        if vl_acc > best_acc:
            best_acc = vl_acc
            model.save_ckpt(best_path)

    print(f"\n  Best val accuracy: {best_acc:.2%} — reloading checkpoint...")
    return MultiHeadModernBERT.load_ckpt(best_path), best_acc

print("✓ Training utilities defined")

### 4A — Clickbait Head
Trains the full model (backbone + clickbait head) for `EPOCHS_TASK` epochs.

In [ ]:
# @title Train clickbait head
model, cb_best = train_task(model, "clickbait", cb_train, cb_val)

### 4B — Political Leaning Head
Continues from the clickbait checkpoint — backbone has already adapted to news text.

In [ ]:
# @title Train political leaning head
model, pol_best = train_task(model, "leaning", pol_train, pol_val)

### 4C — Sentiment Head

In [ ]:
# @title Train sentiment head
model, sent_best = train_task(model, "sentiment", sent_train, sent_val)

### 4D — Multi-Task Fine-Tuning
Trains all three heads simultaneously with a weighted combined loss.
Uses half the learning rate to avoid catastrophic forgetting.

In [ ]:
# @title Multi-task fine-tuning

def train_multitask(model, n_epochs=EPOCHS_MULTI):
    print(f"\n{'='*55}")
    print(f"  MULTI-TASK FINE-TUNING  ({n_epochs} epochs, lr={LR/2:.0e})")
    print(f"{'='*55}")

    optimizer    = AdamW(model.parameters(), lr=LR/2, weight_decay=WEIGHT_DECAY)
    steps_epoch  = max(len(cb_train), len(pol_train), len(sent_train))
    total_steps  = steps_epoch * n_epochs
    scheduler    = get_linear_schedule_with_warmup(optimizer,
                       int(total_steps * WARMUP_RATIO), total_steps)
    scaler       = GradScaler() if USE_FP16 else None

    best_combined = 0.0
    best_path = f"{CKPT_DIR}/best_multitask.pt"

    def _get(it, loader):
        try:    return next(it)
        except: return next(iter(loader))

    for epoch in range(1, n_epochs + 1):
        model.train()
        t0          = time.time()
        total_loss  = 0.0
        cb_it   = iter(cb_train)
        pol_it  = iter(pol_train)
        sent_it = iter(sent_train)

        pbar = tqdm(range(steps_epoch),
                    desc=f"  Multi-task epoch {epoch}/{n_epochs}", leave=False, ncols=90)
        for _ in pbar:
            optimizer.zero_grad(set_to_none=True)

            def fwd(batch, task):
                ids  = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                lbls = batch["labels"].to(device)
                return criterion(model(ids, mask, task=task), lbls)

            cb_b, pol_b, sent_b = (_get(cb_it, cb_train),
                                   _get(pol_it, pol_train),
                                   _get(sent_it, sent_train))

            if scaler:
                with autocast():
                    loss = fwd(cb_b,"clickbait") + fwd(pol_b,"leaning") + fwd(sent_b,"sentiment")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer); scaler.update()
            else:
                loss = fwd(cb_b,"clickbait") + fwd(pol_b,"leaning") + fwd(sent_b,"sentiment")
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            scheduler.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.3f}")

        _, cb_acc,   _, _ = evaluate(model, cb_val,   "clickbait")
        _, pol_acc,  _, _ = evaluate(model, pol_val,  "leaning")
        _, sent_acc, _, _ = evaluate(model, sent_val, "sentiment")
        combined = (cb_acc + pol_acc + sent_acc) / 3
        mins = (time.time() - t0) / 60

        marker = " ★" if combined > best_combined else ""
        print(f"  Epoch {epoch}/{n_epochs}  loss={total_loss/steps_epoch:.4f}  "
              f"CB={cb_acc:.2%}  Lean={pol_acc:.2%}  Sent={sent_acc:.2%}  "
              f"avg={combined:.2%}  {mins:.1f}m{marker}")

        if combined > best_combined:
            best_combined = combined
            model.save_ckpt(best_path)

    print(f"\n  Best combined accuracy: {best_combined:.2%} — reloading checkpoint...")
    return MultiHeadModernBERT.load_ckpt(best_path), best_combined

model, multi_best = train_multitask(model)

## 📈 Section 5 — Evaluation

In [ ]:
# @title Final evaluation on validation sets

TASK_META = {
    "clickbait": {"loader": cb_val,   "names": ["Not Clickbait", "Clickbait"]},
    "leaning":   {"loader": pol_val,  "names": ["Left", "Center", "Right"]},
    "sentiment": {"loader": sent_val, "names": ["Negative", "Neutral", "Positive"]},
}

print("=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

results = {}
for task, meta in TASK_META.items():
    _, acc, preds, true = evaluate(model, meta["loader"], task)
    results[task] = acc
    print(f"\n─── {task.upper()}  (val accuracy: {acc:.2%}) ───")
    print(classification_report(true, preds, target_names=meta["names"], digits=3))

print("=" * 60)
print(f"Summary: clickbait={results['clickbait']:.2%}  "
      f"leaning={results['leaning']:.2%}  "
      f"sentiment={results['sentiment']:.2%}")
print(f"Average: {sum(results.values())/3:.2%}")

# Quick sanity check with real sentences
print("\n" + "=" * 60)
print("SANITY CHECK — manual inference")
print("=" * 60)
TESTS = [
    ("You WON'T BELIEVE what this politician just said!",
     "Experts react as shocking video goes viral nationwide."),
    ("Federal Reserve holds rates steady amid moderate growth",
     "The central bank cited stable inflation data in its decision Wednesday."),
    ("GOP tax cuts for billionaires devastate working families",
     "Republicans push regressive policy that widens wealth inequality."),
    ("Biden's open-border agenda threatens national security",
     "Conservative critics slam the administration's immigration failures."),
]
model.eval()
with torch.no_grad():
    for title, body in TESTS:
        text = f"{title} [SEP] {body}"
        enc  = tokenizer(text, return_tensors="pt",
                         max_length=256, padding="max_length", truncation=True)
        ids  = enc["input_ids"].to(device)
        mask = enc["attention_mask"].to(device)

        out = model(ids, mask, task="all")
        cb_p  = torch.softmax(out["clickbait"], 1)[0, 1].item() * 100
        lean  = torch.softmax(out["leaning"],   1)[0]
        sent  = torch.softmax(out["sentiment"], 1)[0]
        pol_s = (-1*lean[0] + 0*lean[1] + 1*lean[2]).item()
        sen_s = (-1*sent[0] + 0*sent[1] + 1*sent[2]).item()

        print(f"\nTitle: '{title[:60]}'")
        print(f"  Clickbait: {cb_p:.1f}%  |  Political: {pol_s:+.3f}  |  Sentiment: {sen_s:+.3f}")

## 💾 Section 6 — Export & Download

Saves the model in two formats:
1. **HuggingFace format** backbone + `task_heads.pt` — used by `unblur/backend/analyzer.py`
2. **Full checkpoint** `model_full.pt` — for easy reloading / further training

In [ ]:
# @title Export model files

def export_model(model, tokenizer, out_dir=OUTPUT_DIR):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    model.eval()

    print("Exporting model...\n")

    # 1 — Backbone in HuggingFace format (config.json + model.safetensors)
    print("[1/4] Saving backbone (HuggingFace format)...")
    model.backbone.save_pretrained(str(out))
    print(f"      config.json + model.safetensors → {out}/")

    # 2 — Tokenizer
    print("[2/4] Saving tokenizer...")
    tokenizer.save_pretrained(str(out))
    print(f"      tokenizer.json + vocab files → {out}/")

    # 3 — Task heads only (used by analyzer.py's Format 1)
    print("[3/4] Saving task heads (task_heads.pt)...")
    heads_state = {k: v for k, v in model.state_dict().items()
                   if not k.startswith("backbone.")}
    torch.save(heads_state, out / "task_heads.pt")
    print(f"      {len(heads_state)} tensors saved")
    print(f"      Keys: {list(heads_state.keys())[:4]} ...")

    # 4 — Full checkpoint (used by analyzer.py's Format 2)
    print("[4/4] Saving full checkpoint (model_full.pt)...")
    model.save_ckpt(str(out / "model_full.pt"))

    # Model card
    results_str = "\n".join(f"- {k}: {v:.2%}" for k, v in results.items())
    readme = f"""# UnBlur ModernBERT — Multi-Task Model

Fine-tuned from `{MODEL_NAME}`.

## Validation Accuracy
{results_str}

## Files
- `config.json` + `model.safetensors` — backbone weights (HuggingFace format)
- `tokenizer.json` + tokenizer files  — tokenizer
- `task_heads.pt`  — classification head weights only
- `model_full.pt`  — full model checkpoint (backbone + heads)

## Usage
Copy this folder to `unblur/backend/models/` and start the backend.
"""
    (out / "README.md").write_text(readme)

    # File sizes
    print("\nFile sizes:")
    total_mb = 0
    for f in sorted(out.iterdir()):
        if f.is_file():
            mb = f.stat().st_size / 1e6
            total_mb += mb
            print(f"  {f.name:<40} {mb:6.1f} MB")
    print(f"  {'TOTAL':<40} {total_mb:6.1f} MB")
    print(f"\n✓ Export complete: {out}")

export_model(model, tokenizer)

In [ ]:
# @title 📦 Create ZIP and download

ZIP_NAME = "unblur_model.zip"
ZIP_PATH = f"/content/{ZIP_NAME}"

print(f"Creating {ZIP_NAME} ...")
out = Path(OUTPUT_DIR)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for f in sorted(out.rglob("*")):
        if f.is_file():
            arcname = f"unblur_model/{f.relative_to(out)}"
            zf.write(f, arcname)
            print(f"  + {arcname}")

zip_mb = Path(ZIP_PATH).stat().st_size / 1e6
print(f"\n✓ {ZIP_NAME}  ({zip_mb:.0f} MB)")

print("\nStarting download...")
from google.colab import files
files.download(ZIP_PATH)

print("""
╔══════════════════════════════════════════════════════╗
║  ✓ Download complete!                                ║
║                                                      ║
║  Next steps:                                         ║
║  1. Unzip unblur_model.zip                          ║
║  2. Copy contents to unblur/backend/models/          ║
║  3. Run: uvicorn backend.main:app --reload           ║
╚══════════════════════════════════════════════════════╝
""")